# Image Deblurring with Gradient Descent

This notebook demonstrates semi-convergence in image deblurring:
1. Load a test image (256×256)
2. Apply Gaussian blur and add 5% noise
3. Reconstruct using Gradient Descent with exact line search (no regularization)
4. Observe how the error first decreases, then increases as noise is amplified

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import fft2, ifft2
import pandas as pd
from pathlib import Path

# Create figures directory
fig_dir = Path('figures/deblurring')
fig_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

## 1. Load and Prepare Image

We load a photograph, convert to grayscale, crop to square, and resize to 256×256.

In [ ]:
def load_and_prepare_image(path, size=256):
    """Load image, convert to grayscale, crop from right, resize, enhance contrast."""
    img = Image.open(path)
    img_gray = img.convert('L')
    arr = np.array(img_gray, dtype=np.float64)
    
    # Crop square from right side
    h, w = arr.shape
    crop_size = h
    start_w = w - crop_size
    arr_square = arr[:, start_w:]
    
    # Resize
    img_square = Image.fromarray(arr_square.astype(np.uint8))
    img_resized = img_square.resize((size, size), Image.LANCZOS)
    arr_small = np.array(img_resized, dtype=np.float64)
    
    # Normalize and enhance contrast
    arr_small = arr_small - arr_small.min()
    arr_small = arr_small / arr_small.max()
    p2, p98 = np.percentile(arr_small, (2, 98))
    arr_small = np.clip((arr_small - p2) / (p98 - p2), 0, 1)
    
    return arr_small

# Load the test image
x_true = load_and_prepare_image('figures/IMG_6214.jpeg')

plt.figure(figsize=(6, 6))
plt.imshow(x_true, cmap='gray', vmin=0, vmax=1)
plt.title('Original Image (256×256)')
plt.axis('off')
plt.savefig(fig_dir / 'basketball_original.png', dpi=150, bbox_inches='tight', pad_inches=0.1)
plt.show()

## 2. Forward Model: Gaussian Blur + Noise

The forward model is:
$$b = Ax_{\text{true}} + \eta$$
where $A$ is a Gaussian blur operator and $\eta$ is additive Gaussian noise (5% relative).

In [ ]:
def create_gaussian_kernel(size, sigma):
    """Create a Gaussian blur kernel in Fourier domain."""
    fx = np.fft.fftfreq(size)
    fy = np.fft.fftfreq(size)
    FX, FY = np.meshgrid(fx, fy)
    kernel_fft = np.exp(-2 * np.pi**2 * sigma**2 * (FX**2 + FY**2))
    return kernel_fft

def apply_blur(img, kernel_fft):
    """Apply blur using FFT convolution."""
    img_fft = fft2(img)
    blurred_fft = img_fft * kernel_fft
    return np.real(ifft2(blurred_fft))

def apply_blur_transpose(img, kernel_fft):
    """Apply transpose of blur (conjugate in Fourier domain)."""
    img_fft = fft2(img)
    blurred_fft = img_fft * np.conj(kernel_fft)
    return np.real(ifft2(blurred_fft))

# Set up the blur
blur_sigma = 3.0
kernel_fft = create_gaussian_kernel(256, blur_sigma)

# Apply blur
x_blurred = apply_blur(x_true, kernel_fft)

# Add 5% relative noise
noise_level = 0.05
noise = noise_level * np.linalg.norm(x_blurred) / np.sqrt(x_blurred.size) * np.random.randn(*x_blurred.shape)
b = x_blurred + noise

print(f"Noise level: {100 * np.linalg.norm(noise) / np.linalg.norm(x_blurred):.2f}%")

plt.figure(figsize=(6, 6))
plt.imshow(b, cmap='gray', vmin=0, vmax=1)
plt.title('Blurred + 5% Noise')
plt.axis('off')
plt.savefig(fig_dir / 'basketball_blurred_noisy_5pct.png', dpi=150, bbox_inches='tight', pad_inches=0.1)
plt.show()

## 3. Gradient Descent with Exact Line Search

We minimize:
$$\min_x \frac{1}{2}\|Ax - b\|^2$$

using gradient descent with exact line search:
- Gradient: $g = A^T(Ax - b)$
- Exact step size: $\alpha = \frac{\|g\|^2}{\|Ag\|^2}$

In [ ]:
def gd_deblur(b, kernel_fft, x_true, max_iter=5000):
    """
    Gradient Descent with exact line search for deblurring.
    
    Returns:
        x_final: final reconstruction
        x_best: reconstruction with lowest error to ground truth
        best_iter: iteration number of best reconstruction
        history: dict with convergence metrics
    """
    x = np.zeros_like(b)
    
    history = {
        'iteration': [],
        'data_residual': [],
        'error_to_true': [],
        'relative_error': []
    }
    
    x_best = x.copy()
    best_error = np.inf
    best_iter = 0
    x_true_norm = np.linalg.norm(x_true)
    
    for i in range(max_iter):
        # Compute residual and gradient
        Ax = apply_blur(x, kernel_fft)
        r = Ax - b
        g = apply_blur_transpose(r, kernel_fft)
        
        # Exact line search
        Ag = apply_blur(g, kernel_fft)
        g_norm_sq = np.sum(g * g)
        Ag_norm_sq = np.sum(Ag * Ag)
        
        if Ag_norm_sq < 1e-30:
            print(f'Converged at iteration {i+1}')
            break
        
        alpha = g_norm_sq / Ag_norm_sq
        x = x - alpha * g
        
        # Compute metrics
        data_res = np.linalg.norm(r)
        error = np.linalg.norm(x - x_true)
        rel_error = error / x_true_norm
        
        history['iteration'].append(i + 1)
        history['data_residual'].append(data_res)
        history['error_to_true'].append(error)
        history['relative_error'].append(rel_error)
        
        # Track best
        if error < best_error:
            best_error = error
            x_best = x.copy()
            best_iter = i + 1
        
        # Print progress
        if (i + 1) % 1000 == 0 or i == 0:
            print(f'Iter {i+1:4d}: data_res = {data_res:.6e}, rel_error = {rel_error:.4f}')
    
    print(f'\nBest reconstruction at iteration {best_iter}')
    print(f'Best relative error: {best_error/x_true_norm:.4f}')
    
    return x, x_best, best_iter, history

In [ ]:
# Run GD for 5000 iterations
x_final, x_best, best_iter, history = gd_deblur(b, kernel_fft, x_true, max_iter=5000)

## 4. Results and Visualization

In [ ]:
# Save convergence history
df = pd.DataFrame(history)
df.to_csv(fig_dir / 'basketball_gd_convergence_5pct_5k.csv', index=False)
print(f"Saved convergence history to CSV")
df.head(10)

In [ ]:
# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Data residual
axes[0].semilogy(history['iteration'], history['data_residual'], 'b-', linewidth=1.5, label='Data residual')
axes[0].axhline(y=np.linalg.norm(noise), color='r', linestyle='--', label='Noise level')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel(r'$\|Ax - b\|_2$')
axes[0].set_title('Data Residual')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error to ground truth
axes[1].semilogy(history['iteration'], history['error_to_true'], 'g-', linewidth=1.5)
axes[1].axvline(x=best_iter, color='r', linestyle='--', label=f'Best iter = {best_iter}')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel(r'$\|x - x_{true}\|_2$')
axes[1].set_title('Error to Ground Truth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'basketball_gd_convergence_plots_5pct_5k.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save reconstruction images
def save_image(img, filename):
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.axis('off')
    plt.savefig(fig_dir / filename, dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.close()
    print(f'Saved {filename}')

save_image(x_best, f'basketball_gd_reconstruction_best_5pct_iter{best_iter}.png')
save_image(x_final, 'basketball_gd_reconstruction_final_5pct_5k.png')

In [ ]:
# Create comparison figure
rel_err_best = np.linalg.norm(x_best - x_true) / np.linalg.norm(x_true)
rel_err_final = np.linalg.norm(x_final - x_true) / np.linalg.norm(x_true)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(x_true, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(b, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Blurred + 5% Noise')
axes[0, 1].axis('off')

axes[1, 0].imshow(x_best, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title(f'Best GD (iter {best_iter}, rel. err = {rel_err_best:.3f})')
axes[1, 0].axis('off')

axes[1, 1].imshow(x_final, cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title(f'Final GD (iter 5000, rel. err = {rel_err_final:.3f})')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(fig_dir / 'basketball_gd_all_results_5pct_5k.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print summary
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Image size: {x_true.shape}")
print(f"Blur sigma: {blur_sigma}")
print(f"Noise level: {noise_level*100}%")
print(f"")
print(f"Best iteration: {best_iter}")
print(f"Best relative error: {rel_err_best:.4f}")
print(f"Final relative error (iter 5000): {rel_err_final:.4f}")

## Key Observations

1. **Semi-convergence**: GD without regularization exhibits semi-convergence — the error to the true solution initially decreases, reaches a minimum around iteration 43, then steadily increases as high-frequency noise is amplified.

2. **Data residual vs. reconstruction error**: The data residual continues to decrease throughout, but this doesn't correspond to better reconstruction quality after the optimal stopping point.

3. **Need for regularization**: This demonstrates why regularization (Tikhonov, TV, early stopping, or learned priors) is essential for ill-posed inverse problems.

4. **Early stopping as regularization**: The best reconstruction is achieved by stopping early — the iteration number acts as an implicit regularization parameter.